# Workflow evaluation pipeline

- This notebook loads saved score files and runs workflow-level evaluation, threshold tuning, paired instability analysis, disagreement-based deferral, confidence abstention, and supporting model comparisons.
- It expects prediction score files under `./data` or the configured relative input directory.
- See the README for setup and expected inputs.

In [ ]:
# Cell 1: setup + config
import os
import json
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

CONFIG = {
    "seed": 42,
    "task2_root": "./data",
    "output_root": "./outputs",
    "score_files": {
        "english_dev": "predictions/english_dev_scores.csv",
        "english_test": "predictions/english_test_scores.csv",
        "codemix_test": "predictions/codemix_test_scores.csv",
        "tamil_test": "predictions/tamil_test_scores.csv",
    },
    "risk_budget": {
        "epsilon_H": 0.05,
        "epsilon_NH": 0.10
    },
    "threshold_grid": {
        "tau_low_values":  [round(x, 3) for x in np.arange(0.001, 0.201, 0.005)],
        "tau_high_values": [round(x, 3) for x in np.arange(0.80, 0.999, 0.005)]
    }
}

TASK2_ROOT = Path(CONFIG["task2_root"])
ROOT = Path(CONFIG["output_root"])
DIRS = {
    "metrics": ROOT / "metrics",
    "predictions": ROOT / "predictions",
    "logs": ROOT / "logs"
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

print("Task 2 root:", TASK2_ROOT)
print("Task 3-5 root:", ROOT)

Note: During notebook execution, generated metrics, predictions, and logs are written to the `./outputs` directory.

In [ ]:
# Cell 2: load score files
def read_scores(relative_path, name):
    path = TASK2_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. See the README for dataset preparation and input file instructions."
        )
    df = pd.read_csv(path)
    needed = ["sourceid", "label", "prob_hate", "prob_non_hate", "pred_label", "scenario"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    df["label"] = df["label"].astype(int)
    return df.reset_index(drop=True)


english_dev  = read_scores(CONFIG["score_files"]["english_dev"],  "english_dev")
english_test = read_scores(CONFIG["score_files"]["english_test"], "english_test")
codemix_test = read_scores(CONFIG["score_files"]["codemix_test"], "codemix_test")
tamil_test   = read_scores(CONFIG["score_files"]["tamil_test"],   "tamil_test")

print("english_dev :", english_dev.shape)
print("english_test:", english_test.shape)
print("codemix_test:", codemix_test.shape)
print("tamil_test  :", tamil_test.shape)

In [ ]:
# Cell 3: routing + metric helpers
def apply_three_action_routing(df, tau_low, tau_high, score_col="prob_hate"):
    out = df.copy()
    def route(score):
        if score < tau_low:
            return "ALLOW"
        elif score > tau_high:
            return "FLAG"
        else:
            return "REVIEW"
    out["tau_low"] = tau_low
    out["tau_high"] = tau_high
    out["action"] = out[score_col].apply(route)
    return out

def compute_workflow_metrics(df):
    y = df["label"].astype(int).values
    a = df["action"].values
    hate_mask = (y == 1)
    non_hate_mask = (y == 0)

    hfa = float(((a == "ALLOW") & hate_mask).sum() / hate_mask.sum()) if hate_mask.sum() > 0 else np.nan
    nff = float(((a == "FLAG") & non_hate_mask).sum() / non_hate_mask.sum()) if non_hate_mask.sum() > 0 else np.nan
    rr = float((a == "REVIEW").mean())
    allow_rate = float((a == "ALLOW").mean())
    flag_rate = float((a == "FLAG").mean())
    coverage = float(1.0 - rr)

    return {
        "HFA": hfa,
        "NFF": nff,
        "RR": rr,
        "ALLOW_rate": allow_rate,
        "FLAG_rate": flag_rate,
        "Coverage": coverage
    }

def compute_metrics_from_action_columns(df, action_col, label_col="label"):
    y = df[label_col].astype(int).values
    a = df[action_col].values
    hate_mask = (y == 1)
    non_hate_mask = (y == 0)

    hfa = float(((a == "ALLOW") & hate_mask).sum() / hate_mask.sum()) if hate_mask.sum() > 0 else np.nan
    nff = float(((a == "FLAG") & non_hate_mask).sum() / non_hate_mask.sum()) if non_hate_mask.sum() > 0 else np.nan
    rr = float((a == "REVIEW").mean())

    return {
        "HFA": hfa,
        "NFF": nff,
        "RR": rr,
        "ALLOW_rate": float((a == "ALLOW").mean()),
        "FLAG_rate": float((a == "FLAG").mean()),
        "Coverage": float(1.0 - rr)
    }

def threshold_sweep(df, split_name):
    rows = []
    for tau_low in CONFIG["threshold_grid"]["tau_low_values"]:
        for tau_high in CONFIG["threshold_grid"]["tau_high_values"]:
            if tau_low >= tau_high:
                continue
            routed = apply_three_action_routing(df, tau_low, tau_high)
            m = compute_workflow_metrics(routed)
            rows.append({
                "split": split_name,
                "tau_low": tau_low,
                "tau_high": tau_high,
                **m
            })
    return pd.DataFrame(rows)

def pick_best_thresholds(sweep_df, epsilon_H, epsilon_NH):
    feasible = sweep_df[
        (sweep_df["HFA"] <= epsilon_H) &
        (sweep_df["NFF"] <= epsilon_NH)
    ].copy()

    if len(feasible) > 0:
        best = feasible.sort_values(
            by=["RR", "HFA", "NFF", "tau_low", "tau_high"],
            ascending=[True, True, True, True, True]
        ).iloc[0].to_dict()
        best["selection_type"] = "feasible"
        return best

    fallback = sweep_df.copy()
    fallback["hfa_excess"] = (fallback["HFA"] - epsilon_H).clip(lower=0)
    fallback["nff_excess"] = (fallback["NFF"] - epsilon_NH).clip(lower=0)
    fallback["total_excess"] = fallback["hfa_excess"] + fallback["nff_excess"]

    best = fallback.sort_values(
        by=["total_excess", "hfa_excess", "nff_excess", "RR", "tau_low", "tau_high"],
        ascending=[True, True, True, True, True, True]
    ).iloc[0].to_dict()
    best["selection_type"] = "closest_unfeasible"
    return best

def build_pair_df(base_df, alt_df, base_name, alt_name):
    pair_cols = ["sourceid", "label", "prob_hate", "action"]
    left = base_df[pair_cols].copy().rename(columns={
        "prob_hate": f"prob_hate_{base_name}",
        "action": f"action_{base_name}"
    })
    right = alt_df[pair_cols].copy().rename(columns={
        "prob_hate": f"prob_hate_{alt_name}",
        "action": f"action_{alt_name}"
    })
    paired = left.merge(right, on=["sourceid", "label"], how="inner")
    paired["decision_flip"] = (paired[f"action_{base_name}"] != paired[f"action_{alt_name}"]).astype(int)
    return paired

def disagreement_deferral(pair_df, base_name, alt_name, output_action_col="final_action_disagreement"):
    out = pair_df.copy()
    out[output_action_col] = np.where(
        out[f"action_{base_name}"] == out[f"action_{alt_name}"],
        out[f"action_{alt_name}"],
        "REVIEW"
    )
    return out

In [ ]:
# Cell 4: tune thresholds on clean dev only
dev_sweep = threshold_sweep(english_dev, "english_dev")
best_thresholds = pick_best_thresholds(
    dev_sweep,
    epsilon_H=CONFIG["risk_budget"]["epsilon_H"],
    epsilon_NH=CONFIG["risk_budget"]["epsilon_NH"]
)

best_thresholds_df = pd.DataFrame([best_thresholds])
dev_sweep.to_csv(DIRS["metrics"] / "english_dev_threshold_sweep.csv", index=False)
best_thresholds_df.to_csv(DIRS["metrics"] / "best_thresholds.csv", index=False)

print("Chosen thresholds:")
print(best_thresholds_df[["tau_low", "tau_high", "HFA", "NFF", "RR", "selection_type"]].to_string(index=False))

In [ ]:
# Cell 5: apply frozen thresholds to all test views
TAU_LOW = float(best_thresholds["tau_low"])
TAU_HIGH = float(best_thresholds["tau_high"])

english_test_routed = apply_three_action_routing(english_test, TAU_LOW, TAU_HIGH)
codemix_test_routed = apply_three_action_routing(codemix_test, TAU_LOW, TAU_HIGH)
tamil_test_routed   = apply_three_action_routing(tamil_test, TAU_LOW, TAU_HIGH)

english_test_routed.to_csv(DIRS["predictions"] / "english_test_routed.csv", index=False)
codemix_test_routed.to_csv(DIRS["predictions"] / "codemix_test_routed.csv", index=False)
tamil_test_routed.to_csv(DIRS["predictions"] / "tamil_test_routed.csv", index=False)

clean_metrics   = compute_workflow_metrics(english_test_routed)
codemix_metrics = compute_workflow_metrics(codemix_test_routed)
tamil_metrics   = compute_workflow_metrics(tamil_test_routed)

main_results = pd.DataFrame([
    {"setting": "clean_baseline", **clean_metrics},
    {"setting": "codemix_baseline", **codemix_metrics},
    {"setting": "tamil_baseline", **tamil_metrics},
])

main_results.to_csv(DIRS["metrics"] / "main_results_baseline.csv", index=False)
print("Main results:")
print(main_results.to_string(index=False))

In [ ]:
# Cell 6: paired instability metrics for both alternate views
paired_clean_vs_codemix = build_pair_df(
    english_test_routed, codemix_test_routed, "clean", "codemix"
)
paired_clean_vs_tamil = build_pair_df(
    english_test_routed, tamil_test_routed, "clean", "tamil"
)

if len(paired_clean_vs_codemix) != len(english_test_routed):
    print("Warning: paired_clean_vs_codemix merge count differs from english_test count")
if len(paired_clean_vs_tamil) != len(english_test_routed):
    print("Warning: paired_clean_vs_tamil merge count differs from english_test count")

paired_summary = pd.DataFrame([
    {
        "comparison": "clean_vs_codemix",
        "decision_flip_rate": float(paired_clean_vs_codemix["decision_flip"].mean()),
        "delta_HFA": codemix_metrics["HFA"] - clean_metrics["HFA"],
        "delta_NFF": codemix_metrics["NFF"] - clean_metrics["NFF"],
        "clean_HFA": clean_metrics["HFA"],
        "alt_HFA": codemix_metrics["HFA"],
        "clean_NFF": clean_metrics["NFF"],
        "alt_NFF": codemix_metrics["NFF"],
        "clean_RR": clean_metrics["RR"],
        "alt_RR": codemix_metrics["RR"],
    },
    {
        "comparison": "clean_vs_tamil",
        "decision_flip_rate": float(paired_clean_vs_tamil["decision_flip"].mean()),
        "delta_HFA": tamil_metrics["HFA"] - clean_metrics["HFA"],
        "delta_NFF": tamil_metrics["NFF"] - clean_metrics["NFF"],
        "clean_HFA": clean_metrics["HFA"],
        "alt_HFA": tamil_metrics["HFA"],
        "clean_NFF": clean_metrics["NFF"],
        "alt_NFF": tamil_metrics["NFF"],
        "clean_RR": clean_metrics["RR"],
        "alt_RR": tamil_metrics["RR"],
    }
])

paired_clean_vs_codemix.to_csv(DIRS["predictions"] / "paired_clean_vs_codemix_actions.csv", index=False)
paired_clean_vs_tamil.to_csv(DIRS["predictions"] / "paired_clean_vs_tamil_actions.csv", index=False)
paired_summary.to_csv(DIRS["metrics"] / "paired_instability_summary.csv", index=False)

print("Paired instability summary:")
print(paired_summary.to_string(index=False))

In [ ]:
# Cell 6b: Imports for statistical significance testing of paired proportions
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import pandas as pd

In [ ]:
# Cell 6c: Helper functions for McNemar's test, used to analyze paired binary data for significance
def paired_binary_table(df, clean_event, alt_event):
    clean = clean_event.astype(int)
    alt = alt_event.astype(int)

    n00 = int(((clean == 0) & (alt == 0)).sum())
    n01 = int(((clean == 0) & (alt == 1)).sum())
    n10 = int(((clean == 1) & (alt == 0)).sum())
    n11 = int(((clean == 1) & (alt == 1)).sum())

    return np.array([[n00, n01],
                     [n10, n11]])

def run_mcnemar(df, clean_event, alt_event, exact=True, correction=True):
    table = paired_binary_table(df, clean_event, alt_event)
    result = mcnemar(table, exact=exact, correction=correction)
    return {
        "n00": int(table[0, 0]),
        "n01": int(table[0, 1]),
        "n10": int(table[1, 0]),
        "n11": int(table[1, 1]),
        "statistic": float(result.statistic) if result.statistic is not None else np.nan,
        "p_value": float(result.pvalue),
    }

In [ ]:
# Cell 6d: Conducts and displays McNemar's significance tests for HFA, NFF, and RR metrics
def significance_tests_for_pair(pair_df, alt_name):
    out = []

    clean_action_col = "action_clean"
    alt_action_col = f"action_{alt_name}"

    # HFA: among hate rows, was action ALLOW?
    hate_df = pair_df[pair_df["label"] == 1].copy()
    hfa_res = run_mcnemar(
        hate_df,
        clean_event=(hate_df[clean_action_col] == "ALLOW"),
        alt_event=(hate_df[alt_action_col] == "ALLOW"),
        exact=True
    )
    out.append({
        "comparison": f"clean_vs_{alt_name}",
        "metric": "HFA",
        **hfa_res
    })

    # NFF: among non-hate rows, was action FLAG?
    nonhate_df = pair_df[pair_df["label"] == 0].copy()
    nff_res = run_mcnemar(
        nonhate_df,
        clean_event=(nonhate_df[clean_action_col] == "FLAG"),
        alt_event=(nonhate_df[alt_action_col] == "FLAG"),
        exact=True
    )
    out.append({
        "comparison": f"clean_vs_{alt_name}",
        "metric": "NFF",
        **nff_res
    })

    # RR: among all rows, was action REVIEW?
    rr_res = run_mcnemar(
        pair_df,
        clean_event=(pair_df[clean_action_col] == "REVIEW"),
        alt_event=(pair_df[alt_action_col] == "REVIEW"),
        exact=True
    )
    out.append({
        "comparison": f"clean_vs_{alt_name}",
        "metric": "RR",
        **rr_res
    })

    return pd.DataFrame(out)

sig_codemix = significance_tests_for_pair(paired_clean_vs_codemix, "codemix")
sig_tamil = significance_tests_for_pair(paired_clean_vs_tamil, "tamil")

significance_results = pd.concat([sig_codemix, sig_tamil], ignore_index=True)
significance_results.to_csv(DIRS["metrics"] / "paired_significance_results.csv", index=False)
print("Significance results:")
print(significance_results.to_string(index=False))

In [ ]:
# Cell 7: Implements a disagreement-based deferral strategy
defer_codemix = disagreement_deferral(
    paired_clean_vs_codemix, "clean", "codemix", "final_action_disagreement"
)
defer_tamil = disagreement_deferral(
    paired_clean_vs_tamil, "clean", "tamil", "final_action_disagreement"
)

disagreement_metrics_codemix = compute_metrics_from_action_columns(
    defer_codemix, "final_action_disagreement"
)
disagreement_metrics_tamil = compute_metrics_from_action_columns(
    defer_tamil, "final_action_disagreement"
)

disagreement_results = pd.DataFrame([
    {"setting": "codemix_with_disagreement_deferral", **disagreement_metrics_codemix},
    {"setting": "tamil_with_disagreement_deferral", **disagreement_metrics_tamil},
])

defer_codemix.to_csv(DIRS["predictions"] / "paired_with_codemix_disagreement_deferral.csv", index=False)
defer_tamil.to_csv(DIRS["predictions"] / "paired_with_tamil_disagreement_deferral.csv", index=False)
disagreement_results.to_csv(DIRS["metrics"] / "disagreement_results.csv", index=False)

print("Disagreement results:")
print(disagreement_results.to_string(index=False))

In [ ]:
# Cell 8: combined result tables + summary
final_main_table = pd.concat(
    [main_results, disagreement_results],
    ignore_index=True
)
final_main_table.to_csv(DIRS["metrics"] / "final_main_table.csv", index=False)

run_summary = pd.DataFrame([
    {
        "tau_low": TAU_LOW,
        "tau_high": TAU_HIGH,
        "risk_budget_HFA": CONFIG["risk_budget"]["epsilon_H"],
        "risk_budget_NFF": CONFIG["risk_budget"]["epsilon_NH"],
        "clean_vs_codemix_flip_rate": float(paired_clean_vs_codemix["decision_flip"].mean()),
        "clean_vs_tamil_flip_rate": float(paired_clean_vs_tamil["decision_flip"].mean()),
        "delta_HFA_codemix": codemix_metrics["HFA"] - clean_metrics["HFA"],
        "delta_NFF_codemix": codemix_metrics["NFF"] - clean_metrics["NFF"],
        "delta_HFA_tamil": tamil_metrics["HFA"] - clean_metrics["HFA"],
        "delta_NFF_tamil": tamil_metrics["NFF"] - clean_metrics["NFF"],
    }
])
run_summary.to_csv(DIRS["logs"] / "run_summary.csv", index=False)

print("Final main table:")
print(final_main_table.to_string(index=False))

print("\nRun summary:")
print(run_summary.to_string(index=False))

In [ ]:
# Cell 8b: confidence-based abstention baseline

CONF_ROOT = ROOT / "confidence_abstention_baseline"
CONF_DIRS = {
    "metrics": CONF_ROOT / "metrics",
    "predictions": CONF_ROOT / "predictions",
    "logs": CONF_ROOT / "logs",
}
for d in CONF_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def apply_confidence_abstention(df, tau_conf):
    out = df.copy()
    out["confidence"] = out[["prob_hate", "prob_non_hate"]].max(axis=1)
    out["pred_label"] = out["pred_label"].astype(int)

    def route_row(row):
        if row["confidence"] < tau_conf:
            return "REVIEW"
        return "FLAG" if row["pred_label"] == 1 else "ALLOW"

    out["tau_conf"] = tau_conf
    out["action_conf"] = out.apply(route_row, axis=1)
    return out

def confidence_sweep(df, split_name):
    rows = []
    tau_values = [round(x, 3) for x in np.arange(0.50, 1.001, 0.005)]
    for tau_conf in tau_values:
        routed = apply_confidence_abstention(df, tau_conf)
        m = compute_metrics_from_action_columns(routed, "action_conf")
        rows.append({
            "split": split_name,
            "tau_conf": tau_conf,
            **m
        })
    return pd.DataFrame(rows)

def pick_best_conf_threshold(sweep_df, epsilon_H, epsilon_NH):
    feasible = sweep_df[
        (sweep_df["HFA"] <= epsilon_H) &
        (sweep_df["NFF"] <= epsilon_NH)
    ].copy()

    if len(feasible) > 0:
        best = feasible.sort_values(
            by=["RR", "HFA", "NFF", "tau_conf"],
            ascending=[True, True, True, True]
        ).iloc[0].to_dict()
        best["selection_type"] = "feasible"
        return best

    fallback = sweep_df.copy()
    fallback["hfa_excess"] = (fallback["HFA"] - epsilon_H).clip(lower=0)
    fallback["nff_excess"] = (fallback["NFF"] - epsilon_NH).clip(lower=0)
    fallback["total_excess"] = fallback["hfa_excess"] + fallback["nff_excess"]
    best = fallback.sort_values(
        by=["total_excess", "hfa_excess", "nff_excess", "RR", "tau_conf"],
        ascending=[True, True, True, True, True]
    ).iloc[0].to_dict()
    best["selection_type"] = "closest_unfeasible"
    return best

conf_dev_sweep = confidence_sweep(english_dev, "english_dev")
best_conf = pick_best_conf_threshold(
    conf_dev_sweep,
    epsilon_H=CONFIG["risk_budget"]["epsilon_H"],
    epsilon_NH=CONFIG["risk_budget"]["epsilon_NH"]
)
best_conf_df = pd.DataFrame([best_conf])

TAU_CONF = float(best_conf["tau_conf"])

english_test_conf = apply_confidence_abstention(english_test, TAU_CONF)
codemix_test_conf = apply_confidence_abstention(codemix_test, TAU_CONF)
tamil_test_conf = apply_confidence_abstention(tamil_test, TAU_CONF)

clean_conf_metrics = compute_metrics_from_action_columns(english_test_conf, "action_conf")
codemix_conf_metrics = compute_metrics_from_action_columns(codemix_test_conf, "action_conf")
tamil_conf_metrics = compute_metrics_from_action_columns(tamil_test_conf, "action_conf")

confidence_results = pd.DataFrame([
    {"setting": "clean_confidence_abstention", **clean_conf_metrics},
    {"setting": "codemix_confidence_abstention", **codemix_conf_metrics},
    {"setting": "tamil_confidence_abstention", **tamil_conf_metrics},
])

paired_clean_vs_codemix_conf = build_pair_df(
    english_test_conf.rename(columns={"action_conf": "action"}),
    codemix_test_conf.rename(columns={"action_conf": "action"}),
    "clean", "codemix"
)
paired_clean_vs_tamil_conf = build_pair_df(
    english_test_conf.rename(columns={"action_conf": "action"}),
    tamil_test_conf.rename(columns={"action_conf": "action"}),
    "clean", "tamil"
)

confidence_paired_summary = pd.DataFrame([
    {
        "comparison": "clean_vs_codemix_confidence",
        "decision_flip_rate": float(paired_clean_vs_codemix_conf["decision_flip"].mean()),
        "delta_HFA": codemix_conf_metrics["HFA"] - clean_conf_metrics["HFA"],
        "delta_NFF": codemix_conf_metrics["NFF"] - clean_conf_metrics["NFF"],
        "clean_HFA": clean_conf_metrics["HFA"],
        "alt_HFA": codemix_conf_metrics["HFA"],
        "clean_NFF": clean_conf_metrics["NFF"],
        "alt_NFF": codemix_conf_metrics["NFF"],
        "clean_RR": clean_conf_metrics["RR"],
        "alt_RR": codemix_conf_metrics["RR"],
    },
    {
        "comparison": "clean_vs_tamil_confidence",
        "decision_flip_rate": float(paired_clean_vs_tamil_conf["decision_flip"].mean()),
        "delta_HFA": tamil_conf_metrics["HFA"] - clean_conf_metrics["HFA"],
        "delta_NFF": tamil_conf_metrics["NFF"] - clean_conf_metrics["NFF"],
        "clean_HFA": clean_conf_metrics["HFA"],
        "alt_HFA": tamil_conf_metrics["HFA"],
        "clean_NFF": clean_conf_metrics["NFF"],
        "alt_NFF": tamil_conf_metrics["NFF"],
        "clean_RR": clean_conf_metrics["RR"],
        "alt_RR": tamil_conf_metrics["RR"],
    }
])

conf_dev_sweep.to_csv(CONF_DIRS["metrics"] / "english_dev_confidence_sweep.csv", index=False)
best_conf_df.to_csv(CONF_DIRS["metrics"] / "best_confidence_threshold.csv", index=False)
english_test_conf.to_csv(CONF_DIRS["predictions"] / "english_test_confidence_abstention.csv", index=False)
codemix_test_conf.to_csv(CONF_DIRS["predictions"] / "codemix_test_confidence_abstention.csv", index=False)
tamil_test_conf.to_csv(CONF_DIRS["predictions"] / "tamil_test_confidence_abstention.csv", index=False)
confidence_results.to_csv(CONF_DIRS["metrics"] / "confidence_abstention_results.csv", index=False)
confidence_paired_summary.to_csv(CONF_DIRS["metrics"] / "confidence_abstention_paired_summary.csv", index=False)

print("Chosen confidence threshold:")
print(best_conf_df[["tau_conf", "HFA", "NFF", "RR", "selection_type"]].to_string(index=False))

print("\nConfidence-based abstention results:")
print(confidence_results.to_string(index=False))

print("\nConfidence-based paired instability summary:")
print(confidence_paired_summary.to_string(index=False))

In [ ]:
# Cell 9: multi-model config for supporting detector analysis

MODEL_SPECS = {
    "mbert_finetuned": {
        "label": "mBERT_finetuned",
        "score_files": {
            "english_dev": "predictions/english_dev_scores.csv",
            "english_test": "predictions/english_test_scores.csv",
            "codemix_test": "predictions/codemix_test_scores.csv",
            "tamil_test": "predictions/tamil_test_scores.csv",
        },
    },
    "hatebert_zeroshot": {
        "label": "HateBERT_zeroshot",
        "score_files": {
            "english_dev": "predictions/hatebert_zeroshot_dev_scores.csv",
            "english_test": "predictions/hatebert_zeroshot_english_test_scores.csv",
            "codemix_test": "predictions/hatebert_zeroshot_codemix_test_scores.csv",
            "tamil_test": "predictions/hatebert_zeroshot_tamil_test_scores.csv",
        },
    },
    "metahatebert_zeroshot": {
        "label": "MetaHateBERT_zeroshot",
        "score_files": {
            "english_dev": "predictions/metahatebert_zeroshot_dev_scores.csv",
            "english_test": "predictions/metahatebert_zeroshot_english_test_scores.csv",
            "codemix_test": "predictions/metahatebert_zeroshot_codemix_test_scores.csv",
            "tamil_test": "predictions/metahatebert_zeroshot_tamil_test_scores.csv",
        },
    },
}

SUPPORT_ROOT = ROOT / "supporting_detector_analysis"
SUPPORT_DIRS = {
    "metrics": SUPPORT_ROOT / "metrics",
    "predictions": SUPPORT_ROOT / "predictions",
    "logs": SUPPORT_ROOT / "logs",
}
for d in SUPPORT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Supporting analysis root:", SUPPORT_ROOT)
print("Models:", list(MODEL_SPECS.keys()))

In [ ]:
# Cell 10: reusable runner for one model
def run_single_model_analysis(model_key, model_spec):
    label = model_spec["label"]

    dev_df      = read_scores(model_spec["score_files"]["english_dev"],  f"{label}_english_dev")
    english_df  = read_scores(model_spec["score_files"]["english_test"], f"{label}_english_test")
    codemix_df  = read_scores(model_spec["score_files"]["codemix_test"], f"{label}_codemix_test")
    tamil_df    = read_scores(model_spec["score_files"]["tamil_test"],   f"{label}_tamil_test")

    dev_sweep_local = threshold_sweep(dev_df, f"{label}_english_dev")
    best_local = pick_best_thresholds(
        dev_sweep_local,
        epsilon_H=CONFIG["risk_budget"]["epsilon_H"],
        epsilon_NH=CONFIG["risk_budget"]["epsilon_NH"]
    )

    tau_low = float(best_local["tau_low"])
    tau_high = float(best_local["tau_high"])

    english_routed = apply_three_action_routing(english_df, tau_low, tau_high)
    codemix_routed = apply_three_action_routing(codemix_df, tau_low, tau_high)
    tamil_routed   = apply_three_action_routing(tamil_df, tau_low, tau_high)

    english_metrics = compute_workflow_metrics(english_routed)
    codemix_metrics = compute_workflow_metrics(codemix_routed)
    tamil_metrics   = compute_workflow_metrics(tamil_routed)

    paired_clean_vs_codemix = build_pair_df(english_routed, codemix_routed, "clean", "codemix")
    paired_clean_vs_tamil   = build_pair_df(english_routed, tamil_routed,   "clean", "tamil")

    flip_rate_codemix = float(paired_clean_vs_codemix["decision_flip"].mean())
    flip_rate_tamil   = float(paired_clean_vs_tamil["decision_flip"].mean())

    delta_hfa_codemix = codemix_metrics["HFA"] - english_metrics["HFA"]
    delta_nff_codemix = codemix_metrics["NFF"] - english_metrics["NFF"]
    delta_hfa_tamil   = tamil_metrics["HFA"] - english_metrics["HFA"]
    delta_nff_tamil   = tamil_metrics["NFF"] - english_metrics["NFF"]

    defer_codemix = disagreement_deferral(
        paired_clean_vs_codemix, "clean", "codemix", "final_action_disagreement"
    )
    defer_tamil = disagreement_deferral(
        paired_clean_vs_tamil, "clean", "tamil", "final_action_disagreement"
    )

    disagreement_metrics_codemix = compute_metrics_from_action_columns(
        defer_codemix, "final_action_disagreement"
    )
    disagreement_metrics_tamil = compute_metrics_from_action_columns(
        defer_tamil, "final_action_disagreement"
    )

    main_rows = pd.DataFrame([
        {"model": label, "setting": "clean_baseline", **english_metrics},
        {"model": label, "setting": "codemix_baseline", **codemix_metrics},
        {"model": label, "setting": "tamil_baseline", **tamil_metrics},
        {"model": label, "setting": "codemix_with_disagreement_deferral", **disagreement_metrics_codemix},
        {"model": label, "setting": "tamil_with_disagreement_deferral", **disagreement_metrics_tamil},
    ])

    instability_rows = pd.DataFrame([
        {
            "model": label,
            "comparison": "clean_vs_codemix",
            "tau_low": tau_low,
            "tau_high": tau_high,
            "risk_budget_HFA": CONFIG["risk_budget"]["epsilon_H"],
            "risk_budget_NFF": CONFIG["risk_budget"]["epsilon_NH"],
            "decision_flip_rate": flip_rate_codemix,
            "delta_HFA": delta_hfa_codemix,
            "delta_NFF": delta_nff_codemix,
            "clean_HFA": english_metrics["HFA"],
            "alt_HFA": codemix_metrics["HFA"],
            "clean_NFF": english_metrics["NFF"],
            "alt_NFF": codemix_metrics["NFF"],
            "clean_RR": english_metrics["RR"],
            "alt_RR": codemix_metrics["RR"],
        },
        {
            "model": label,
            "comparison": "clean_vs_tamil",
            "tau_low": tau_low,
            "tau_high": tau_high,
            "risk_budget_HFA": CONFIG["risk_budget"]["epsilon_H"],
            "risk_budget_NFF": CONFIG["risk_budget"]["epsilon_NH"],
            "decision_flip_rate": flip_rate_tamil,
            "delta_HFA": delta_hfa_tamil,
            "delta_NFF": delta_nff_tamil,
            "clean_HFA": english_metrics["HFA"],
            "alt_HFA": tamil_metrics["HFA"],
            "clean_NFF": english_metrics["NFF"],
            "alt_NFF": tamil_metrics["NFF"],
            "clean_RR": english_metrics["RR"],
            "alt_RR": tamil_metrics["RR"],
        },
    ])

    dev_sweep_local.to_csv(SUPPORT_DIRS["metrics"] / f"{model_key}_english_dev_threshold_sweep.csv", index=False)
    pd.DataFrame([best_local]).to_csv(SUPPORT_DIRS["metrics"] / f"{model_key}_best_thresholds.csv", index=False)

    english_routed.to_csv(SUPPORT_DIRS["predictions"] / f"{model_key}_english_test_routed.csv", index=False)
    codemix_routed.to_csv(SUPPORT_DIRS["predictions"] / f"{model_key}_codemix_test_routed.csv", index=False)
    tamil_routed.to_csv(SUPPORT_DIRS["predictions"] / f"{model_key}_tamil_test_routed.csv", index=False)

    paired_clean_vs_codemix.to_csv(
        SUPPORT_DIRS["predictions"] / f"{model_key}_paired_clean_vs_codemix_actions.csv", index=False
    )
    paired_clean_vs_tamil.to_csv(
        SUPPORT_DIRS["predictions"] / f"{model_key}_paired_clean_vs_tamil_actions.csv", index=False
    )

    defer_codemix.to_csv(
        SUPPORT_DIRS["predictions"] / f"{model_key}_paired_with_codemix_disagreement_deferral.csv", index=False
    )
    defer_tamil.to_csv(
        SUPPORT_DIRS["predictions"] / f"{model_key}_paired_with_tamil_disagreement_deferral.csv", index=False
    )

    main_rows.to_csv(SUPPORT_DIRS["metrics"] / f"{model_key}_main_results.csv", index=False)
    instability_rows.to_csv(SUPPORT_DIRS["metrics"] / f"{model_key}_paired_instability_summary.csv", index=False)

    return {
        "model_key": model_key,
        "model_label": label,
        "best_thresholds": best_local,
        "main_rows": main_rows,
        "instability_rows": instability_rows,
    }

In [ ]:
# Cell 11: run supporting detector analysis for all available models

all_model_outputs = []

for model_key, model_spec in MODEL_SPECS.items():
    try:
        result = run_single_model_analysis(model_key, model_spec)
        all_model_outputs.append(result)
        print(f"Done: {model_spec['label']}")
    except Exception as e:
        print(f"Skipped {model_key} -> {e}")

In [ ]:
# Cell 12: combine model-level outputs into paper-friendly supporting tables
supporting_main_table = pd.concat(
    [x["main_rows"] for x in all_model_outputs],
    ignore_index=True
)

supporting_instability_table = pd.concat(
    [x["instability_rows"] for x in all_model_outputs],
    ignore_index=True
)

supporting_detector_ablation = supporting_instability_table[
    [
        "model",
        "comparison",
        "tau_low",
        "tau_high",
        "decision_flip_rate",
        "delta_HFA",
        "delta_NFF",
        "clean_HFA",
        "alt_HFA",
        "clean_NFF",
        "alt_NFF",
        "clean_RR",
        "alt_RR",
    ]
].sort_values(["model", "comparison"]).reset_index(drop=True)

supporting_main_table.to_csv(
    SUPPORT_DIRS["metrics"] / "supporting_main_table_all_models.csv", index=False
)
supporting_instability_table.to_csv(
    SUPPORT_DIRS["metrics"] / "supporting_instability_all_models.csv", index=False
)
supporting_detector_ablation.to_csv(
    SUPPORT_DIRS["metrics"] / "supporting_detector_ablation.csv", index=False
)

print("Supporting detector ablation:")
print(supporting_detector_ablation.to_string(index=False))

In [ ]:
# Cell 13: quick check of whether alternate views raise risk across models
supporting_detector_ablation["alt_increases_HFA"] = (
    supporting_detector_ablation["alt_HFA"] > supporting_detector_ablation["clean_HFA"]
)
supporting_detector_ablation["alt_increases_NFF"] = (
    supporting_detector_ablation["alt_NFF"] > supporting_detector_ablation["clean_NFF"]
)

quick_check = supporting_detector_ablation[
    ["model", "comparison", "alt_increases_HFA", "alt_increases_NFF", "decision_flip_rate"]
]

quick_check.to_csv(SUPPORT_DIRS["logs"] / "supporting_detector_quick_check.csv", index=False)
print("Quick check of alternate views raising risk:")
print(quick_check.to_string(index=False))

In [ ]:
# Cell 14: Performs bootstrap resampling to calculate confidence intervals

import numpy as np
import pandas as pd
from sklearn.utils import resample

def bootstrap_metrics(df, n_iterations=1000):
    stats = []
    for i in range(n_iterations):
        sample = resample(df, replace=True, random_state=i)
        stats.append(compute_workflow_metrics(sample))

    bootstrap_df = pd.DataFrame(stats)
    summary = bootstrap_df.apply(lambda x: pd.Series({
        'mean': x.mean(),
        'ci_lower': np.percentile(x, 2.5),
        'ci_upper': np.percentile(x, 97.5)
    })).T
    return summary

def bootstrap_deltas_and_flips(pair_df, base_name, alt_name, n_iterations=1000):
    delta_stats = []
    flip_stats = []

    for i in range(n_iterations):
        sample = resample(pair_df, replace=True, random_state=i)

        y = sample['label'].astype(int).values
        a_base = sample[f'action_{base_name}'].values
        a_alt = sample[f'action_{alt_name}'].values

        hate_mask = (y == 1)
        non_hate_mask = (y == 0)

        def get_hfa(a, mask): return float(((a == 'ALLOW') & mask).sum() / mask.sum()) if mask.sum() > 0 else np.nan
        def get_nff(a, mask): return float(((a == 'FLAG') & mask).sum() / mask.sum()) if mask.sum() > 0 else np.nan

        hfa_base = get_hfa(a_base, hate_mask)
        hfa_alt = get_hfa(a_alt, hate_mask)
        nff_base = get_nff(a_base, non_hate_mask)
        nff_alt = get_nff(a_alt, non_hate_mask)

        # Correctly calculate RR inside the bootstrap loop
        rr_base = float((a_base == 'REVIEW').mean())
        rr_alt = float((a_alt == 'REVIEW').mean())

        delta_stats.append({
            'delta_HFA': hfa_alt - hfa_base,
            'delta_NFF': nff_alt - nff_base,
            'delta_RR': rr_alt - rr_base
        })

        flip_stats.append({
            'flip_rate': float((sample[f'action_{base_name}'] != sample[f'action_{alt_name}']).mean())
        })

    delta_summary = pd.DataFrame(delta_stats).apply(lambda x: pd.Series({
        'mean': x.mean(),
        'ci_lower': np.percentile(x, 2.5),
        'ci_upper': np.percentile(x, 97.5)
    })).T

    flip_summary = pd.DataFrame(flip_stats).apply(lambda x: pd.Series({
        'mean': x.mean(),
        'ci_lower': np.percentile(x, 2.5),
        'ci_upper': np.percentile(x, 97.5)
    })).T

    return delta_summary, flip_summary

# 1. Load routed data
eng_routed = pd.read_csv(DIRS['predictions'] / 'english_test_routed.csv')
cm_routed = pd.read_csv(DIRS['predictions'] / 'codemix_test_routed.csv')
tam_routed = pd.read_csv(DIRS['predictions'] / 'tamil_test_routed.csv')

# 2. Load paired data
paired_cm = pd.read_csv(DIRS['predictions'] / 'paired_clean_vs_codemix_actions.csv')
paired_tam = pd.read_csv(DIRS['predictions'] / 'paired_clean_vs_tamil_actions.csv')

# 3. Bootstrap Main Metrics
print("Bootstrapping main metrics...")
res_eng = bootstrap_metrics(eng_routed).assign(setting='clean_baseline')
res_cm = bootstrap_metrics(cm_routed).assign(setting='codemix_baseline')
res_tam = bootstrap_metrics(tam_routed).assign(setting='tamil_baseline')
main_results_ci = pd.concat([res_eng, res_cm, res_tam]).reset_index().rename(columns={'index': 'metric'})

# 4. Bootstrap Deltas and Flips
print("Bootstrapping deltas and flips...")
delta_cm, flip_cm = bootstrap_deltas_and_flips(paired_cm, 'clean', 'codemix')
delta_tam, flip_tam = bootstrap_deltas_and_flips(paired_tam, 'clean', 'tamil')

paired_deltas_ci = pd.concat([
    delta_cm.assign(comparison='clean_vs_codemix'),
    delta_tam.assign(comparison='clean_vs_tamil')
]).reset_index().rename(columns={'index': 'metric'})

flip_rates_ci = pd.concat([
    flip_cm.assign(comparison='clean_vs_codemix'),
    flip_tam.assign(comparison='clean_vs_tamil')
]).reset_index().rename(columns={'index': 'metric'})

# 5. Save results
main_results_ci.to_csv(DIRS['metrics'] / 'main_results_with_ci.csv', index=False)
paired_deltas_ci.to_csv(DIRS['metrics'] / 'paired_deltas_with_ci.csv', index=False)
flip_rates_ci.to_csv(DIRS['metrics'] / 'flip_rates_with_ci.csv', index=False)

print("Bootstrap analysis complete. Files saved to:", DIRS['metrics'])
print("Main results with CI:")
print(main_results_ci.head().to_string(index=False))
print("\nPaired deltas with CI:")
print(paired_deltas_ci.to_string(index=False))
print("\nFlip rates with CI:")
print(flip_rates_ci.to_string(index=False))

In [ ]:
# Cell 15: Visualizes the trade-off between Review Rate (RR) and risk metrics (HFA, NFF).
import matplotlib.pyplot as plt

# Prepare data from the final main table
plot_df = final_main_table.copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

settings = plot_df['setting'].unique()
colors = plt.cm.get_cmap('tab10', len(settings))
markers = ['o', 's', 'D', '^', 'v']

# Helper to plot frontier (lower left is better)
def plot_frontier(ax, x_vals, y_vals):
    # Sort by x to build frontier
    pts = sorted(zip(x_vals, y_vals))
    frontier = [pts[0]]
    for p in pts[1:]:
        if p[1] < frontier[-1][1]:
            frontier.append(p)
    fx, fy = zip(*frontier)
    ax.plot(fx, fy, '--', color='gray', alpha=0.5, label='Efficiency Frontier')

# Plot 1: RR vs HFA
for i, setting in enumerate(settings):
    row = plot_df[plot_df['setting'] == setting]
    ax1.scatter(row['RR'], row['HFA'], color=colors(i), marker=markers[i % len(markers)], s=100, label=setting)

plot_frontier(ax1, plot_df['RR'], plot_df['HFA'])
ax1.set_xlabel('Review Rate (RR)')
ax1.set_ylabel('Hate False Allowance (HFA)')
ax1.set_title('Frontier: RR vs HFA')
ax1.axhline(y=CONFIG['risk_budget']['epsilon_H'], color='r', linestyle=':', label='Budget (e_H)')
ax1.grid(True, alpha=0.3)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 2: RR vs NFF
for i, setting in enumerate(settings):
    row = plot_df[plot_df['setting'] == setting]
    ax2.scatter(row['RR'], row['NFF'], color=colors(i), marker=markers[i % len(markers)], s=100)

plot_frontier(ax2, plot_df['RR'], plot_df['NFF'])
ax2.set_xlabel('Review Rate (RR)')
ax2.set_ylabel('Non-Hate False Flag (NFF)')
ax2.set_title('Frontier: RR vs NFF')
ax2.axhline(y=CONFIG['risk_budget']['epsilon_NH'], color='r', linestyle=':', label='Budget (e_NH)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Advanced Pareto Frontier Visualization.
import matplotlib.pyplot as plt

# Set global font sizes for consistency
plt.rcParams['font.size'] = 10 # Base font size
title_fontsize = 14
axis_label_fontsize = 12
tick_legend_fontsize = 10

# We use the sweep data from English Dev to find the 'Optimal' frontier
sweep_data = pd.read_csv(DIRS['metrics'] / 'english_dev_threshold_sweep.csv')

def get_frontier(df, x_col='RR', y_col='HFA'):
    # Sort by X, find points where Y is strictly decreasing
    sorted_df = df.sort_values(x_col)
    frontier = []
    min_y = float('inf')
    for _, row in sorted_df.iterrows():
        if row[y_col] < min_y:
            frontier.append(row)
            min_y = row[y_col]
    return pd.DataFrame(frontier)

# Define the mapping for the legend labels
legend_label_map = {
    'clean_baseline': 'Clean',
    'codemix_baseline': 'Code-mix',
    'tamil_baseline': 'Tamil',
    'codemix_with_disagreement_deferral': 'Code-mix + disagreement',
    'tamil_with_disagreement_deferral': 'Tamil + disagreement',
}

# Define color map for points
point_colors = {
    'clean_baseline': 'blue',
    'codemix_baseline': 'orange',
    'tamil_baseline': 'green',
    'codemix_with_disagreement_deferral': 'red',
    'tamil_with_disagreement_deferral': 'purple',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6)) # Reverted figure height

# Frontier 1: RR vs HFA
frontier_hfa = get_frontier(sweep_data, 'RR', 'HFA')
ax1.plot(frontier_hfa['RR'], frontier_hfa['HFA'], color='black', linewidth=2.0, label='Threshold frontier')
ax1.fill_between(frontier_hfa['RR'], frontier_hfa['HFA'], 1, color='gray', alpha=0.02, label='Feasible Region') # Add label for legend

# Overlay specific settings
for _, row in final_main_table.iterrows():
    label = legend_label_map.get(row['setting'], row['setting'])
    color = point_colors.get(row['setting'], 'gray') # Get color from map, default to gray

    # Highlight 'clean_baseline' point
    s_val = 100 if row['setting'] == 'clean_baseline' else 65
    edgecolor_val = 'red' if row['setting'] == 'clean_baseline' else 'black'
    linewidth_val = 1.5 if row['setting'] == 'clean_baseline'] else 0.7
    zorder_val = 6 if row['setting'] == 'clean_baseline' else 5

    ax1.scatter(row['RR'], row['HFA'], s=s_val, label=label, color=color, edgecolor=edgecolor_val, linewidth=linewidth_val, zorder=zorder_val) # Points styling
    # For ax2, plot without label, but apply same highlighting logic, as labels will come from ax1's legend data
    ax2.scatter(row['RR'], row['NFF'], s=s_val, color=color, edgecolor=edgecolor_val, linewidth=linewidth_val, zorder=zorder_val)

ax1.axhline(y=CONFIG['risk_budget']['epsilon_H'], color='red', linestyle='--', linewidth=1.0, label='Risk budget') # Budget line: thinner linewidth
ax1.set_title('Review rate vs. hate false-accept', fontsize=title_fontsize)
ax1.set_xlabel('Review Rate', fontsize=axis_label_fontsize)
ax1.set_ylabel('Hate false-accept (HFA)', fontsize=axis_label_fontsize)
ax1.set_ylim(0, 0.2) # Set Y-axis scale
ax1.tick_params(axis='both', which='major', labelsize=tick_legend_fontsize)
ax1.grid(True, alpha=0.2, color='lightgray') # Gridlines

# Frontier 2: RR vs NFF
frontier_nff = get_frontier(sweep_data, 'RR', 'NFF')
# Plot ax2 elements without labels, as legend handles/labels will be shared from ax1
ax2.plot(frontier_nff['RR'], frontier_nff['NFF'], color='black', linewidth=2.0)
ax2.fill_between(frontier_nff['RR'], frontier_nff['NFF'], 1, color='gray', alpha=0.02)
ax2.axhline(y=CONFIG['risk_budget']['epsilon_NH'], color='red', linestyle='--', linewidth=1.0)
ax2.set_title('Review rate vs. non-hate false-flag', fontsize=title_fontsize)
ax2.set_xlabel('Review Rate', fontsize=axis_label_fontsize)
ax2.set_ylabel('Non-hate false-flag (NFF)', fontsize=axis_label_fontsize)
ax2.set_ylim(0, 0.2) # Set Y-axis scale
ax2.tick_params(axis='both', which='major', labelsize=tick_legend_fontsize)
ax2.grid(True, alpha=0.2, color='lightgray') # Gridlines

# --- Shared Legend below both panels ---
# Collect handles and labels from ax1 (which has all the necessary labels)
handles, labels = ax1.get_legend_handles_labels()

# Reorder legend: put 'Clean' at the beginning
clean_label = legend_label_map.get('clean_baseline', 'clean_baseline')
try:
    clean_idx = labels.index(clean_label)
    clean_handle = handles.pop(clean_idx)
    clean_label_val = labels.pop(clean_idx)
    handles.insert(0, clean_handle)
    labels.insert(0, clean_label_val)
except ValueError:
    pass # 'Clean' label not found, proceed with existing order

# Place the figure legend below both subplots
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 0.12), fontsize=tick_legend_fontsize)

# Adjust layout to make space for the legend at the bottom
plt.tight_layout(rect=[0, 0.2, 1, 0.95]) # [left, bottom, right, top] coordinates for the subplot area
plt.show()

In [ ]:
# Cell 17: Computes and displays transition matrices for actions between the clean and codemix views.
def transition_matrix(pair_df, from_col, to_col, normalize=False):
    mat = pd.crosstab(pair_df[from_col], pair_df[to_col], normalize='index' if normalize else False)
    order = ["ALLOW", "REVIEW", "FLAG"]
    mat = mat.reindex(index=order, columns=order, fill_value=0)
    return mat

# Main clean -> codemix transitions
cm_counts = transition_matrix(paired_clean_vs_codemix, "action_clean", "action_codemix", normalize=False)
cm_rowpct = transition_matrix(paired_clean_vs_codemix, "action_clean", "action_codemix", normalize=True)

# Label-conditioned
cm_counts_hate = transition_matrix(
    paired_clean_vs_codemix[paired_clean_vs_codemix["label"] == 1],
    "action_clean", "action_codemix", normalize=False
)
cm_rowpct_hate = transition_matrix(
    paired_clean_vs_codemix[paired_clean_vs_codemix["label"] == 1],
    "action_clean", "action_codemix", normalize=True
)

cm_counts_nonhate = transition_matrix(
    paired_clean_vs_codemix[paired_clean_vs_codemix["label"] == 0],
    "action_clean", "action_codemix", normalize=False
)
cm_rowpct_nonhate = transition_matrix(
    paired_clean_vs_codemix[paired_clean_vs_codemix["label"] == 0],
    "action_clean", "action_codemix", normalize=True
)

cm_counts.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_counts.csv")
cm_rowpct.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_rowpct.csv")
cm_counts_hate.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_counts_hate.csv")
cm_rowpct_hate.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_rowpct_hate.csv")
cm_counts_nonhate.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_counts_nonhate.csv")
cm_rowpct_nonhate.to_csv(DIRS["metrics"] / "clean_to_codemix_transition_rowpct_nonhate.csv")

print("Clean to Codemix Transition Counts:")
print(cm_counts.to_string())

print("Clean to Codemix Transition Row Percentages:")
print(cm_rowpct.round(3).to_string())

print("Clean to Codemix Transition Row Percentages (Hate Speech):")
print(cm_rowpct_hate.round(3).to_string())

print("Clean to Codemix Transition Row Percentages (Non-Hate Speech):")
print(cm_rowpct_nonhate.round(3).to_string())

In [ ]:
# Cell 18: Performs qualitative analysis
text_cols = ["sourceid", "label", "textenglish", "textcodemix"]
text_df = english_test[text_cols].copy()

qual_df = paired_clean_vs_codemix.merge(
    text_df, on=["sourceid", "label"], how="left"
)

cand_allow_flag_nonhate = qual_df[
    (qual_df["label"] == 0) &
    (qual_df["action_clean"] == "ALLOW") &
    (qual_df["action_codemix"] == "FLAG")
]

cand_review_flag_nonhate = qual_df[
    (qual_df["label"] == 0) &
    (qual_df["action_clean"] == "REVIEW") &
    (qual_df["action_codemix"] == "FLAG")
]

cand_allow_review_hate = qual_df[
    (qual_df["label"] == 1) &
    (qual_df["action_clean"] == "ALLOW") &
    (qual_df["action_codemix"] == "REVIEW")
]

cand_allow_flag_hate = qual_df[
    (qual_df["label"] == 1) &
    (qual_df["action_clean"] == "ALLOW") &
    (qual_df["action_codemix"] == "FLAG")
]

print("\nExamples of non-hate text where action changes from ALLOW to FLAG:")
print(cand_allow_flag_nonhate[["sourceid","textenglish","textcodemix","action_clean","action_codemix"]].head(10).to_string(index=False))

print("\nExamples of non-hate text where action changes from REVIEW to FLAG:")
print(cand_review_flag_nonhate[["sourceid","textenglish","textcodemix","action_clean","action_codemix"]].head(10).to_string(index=False))

print("\nExamples of hate text where action changes from ALLOW to REVIEW:")
print(cand_allow_review_hate[["sourceid","textenglish","textcodemix","action_clean","action_codemix"]].head(10).to_string(index=False))

print("\nExamples of hate text where action changes from ALLOW to FLAG:")
print(cand_allow_flag_hate[["sourceid","textenglish","textcodemix","action_clean","action_codemix"]].head(10).to_string(index=False))